# SCADA Flow - Data Cleaning & Analysis

**mcom technology** | © 2026 All rights reserved.

**Source:** `SCADA_FLOW_2025` (sensor logs), `SCADA_FLOW_2026` (device registry, alerts, totalizers)

## Data Quality Issues Addressed
| Issue | Impact | Resolution |
|---|---|---|
| IMEI whitespace (13K rows) | Join mismatches | TRIM() |
| Bogus timestamps (5,122 rows) | Skewed analysis | Filter 2020-2026 |
| P1-P30 as STRING | Can't aggregate | Cast to DOUBLE |
| 25,986 duplicates | Inflated counts | ROW_NUMBER() dedup |
| HTML entities in alerts | Parse issues | REPLACE + REGEXP_EXTRACT |
| PNAME casing variants | Fragmented groups | CASE normalization |

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.scada_cleaned

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.scada_cleaned.device_list AS
SELECT
  TRIM(IDD) AS device_id, UID AS uid,
  TRIM(DEVIEC_NAME) AS device_name, TRIM(SHORT_NAME) AS short_name,
  TRY_CAST(INS_DATE AS DATE) AS installation_date,
  TRIM(IMEI_NUMBER) AS imei, TRIM(SIM_NUMBER) AS sim_number,
  TRY_CAST(SERVICE_DATE AS DATE) AS service_expiry_date,
  TRIM(STATUS) AS status, TRIM(OTHER_1) AS application_type,
  TRY_CAST(OTHER_2 AS DOUBLE) AS latitude,
  TRY_CAST(OTHER_3 AS DOUBLE) AS longitude,
  TRIM(OTHER_4) AS location_name, TRIM(OTHER_5) AS site_code
FROM workspace.scada_flow_2026.tb_device_list

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.scada_cleaned.device_parameters AS
SELECT
  TRIM(IDD) AS param_id, UID AS uid, TRIM(IMEI) AS imei,
  PINDEX AS parameter_index, TRIM(PDN) AS parameter_name,
  TRIM(SN) AS short_code, TRIM(IM) AS mapping_code,
  TRIM(AF) AS alert_flag, TRIM(UNIT) AS unit,
  TRY_CAST(LOW AS DOUBLE) AS low_threshold,
  TRY_CAST(MAX AS DOUBLE) AS high_threshold,
  TRIM(TYPE) AS param_type, TRIM(STATUS) AS status
FROM workspace.scada_flow_2026.tb_device_list_parameter

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.scada_cleaned.sensor_log AS
WITH deduped AS (
  SELECT
    TRIM(IMEI) AS imei, WAY AS data_way, DATA_STAMP AS data_timestamp,
    TRY_CAST(P1 AS DOUBLE) AS p1, TRY_CAST(P2 AS DOUBLE) AS p2,
    TRY_CAST(P3 AS DOUBLE) AS p3, TRY_CAST(P4 AS DOUBLE) AS p4,
    TRY_CAST(P5 AS DOUBLE) AS p5, TRY_CAST(P6 AS DOUBLE) AS p6,
    TRY_CAST(P7 AS DOUBLE) AS p7, TRY_CAST(P8 AS DOUBLE) AS p8,
    TRY_CAST(P9 AS DOUBLE) AS p9, TRY_CAST(P10 AS DOUBLE) AS p10,
    TRY_CAST(P11 AS DOUBLE) AS p11, TRY_CAST(P12 AS DOUBLE) AS p12,
    TRY_CAST(P13 AS DOUBLE) AS p13, TRY_CAST(P14 AS DOUBLE) AS p14,
    TRY_CAST(P15 AS DOUBLE) AS p15, TRY_CAST(P16 AS DOUBLE) AS p16,
    TRY_CAST(P17 AS DOUBLE) AS p17, TRY_CAST(P18 AS DOUBLE) AS p18,
    TRY_CAST(P19 AS DOUBLE) AS p19, TRY_CAST(P20 AS DOUBLE) AS p20,
    TRY_CAST(P21 AS DOUBLE) AS p21, TRY_CAST(P22 AS DOUBLE) AS p22,
    TRY_CAST(P23 AS DOUBLE) AS p23, TRY_CAST(P24 AS DOUBLE) AS p24,
    TRY_CAST(P25 AS DOUBLE) AS p25, TRY_CAST(P26 AS DOUBLE) AS p26,
    TRY_CAST(P27 AS DOUBLE) AS p27, TRY_CAST(P28 AS DOUBLE) AS p28,
    TRY_CAST(P29 AS DOUBLE) AS p29, TRY_CAST(P30 AS DOUBLE) AS p30,
    SERVER_DT AS server_timestamp,
    ROW_NUMBER() OVER (
      PARTITION BY TRIM(IMEI), DATA_STAMP, WAY
      ORDER BY SERVER_DT DESC
    ) AS rn
  FROM workspace.scada_flow_2025.tb_logedata
  WHERE YEAR(DATA_STAMP) BETWEEN 2020 AND 2026
)
SELECT * EXCEPT(rn) FROM deduped WHERE rn = 1

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.scada_cleaned.alerts AS
SELECT
  TRIM(IDD) AS alert_id, REPORT_DATE AS report_date,
  TRIM(ALERT_NAME) AS alert_type,
  TRY_CAST(ALERT_VALUE AS DOUBLE) AS alert_value,
  REPLACE(ALERT_RANGE, '&nbsp;', ' ') AS alert_range_raw,
  TRY_CAST(REGEXP_EXTRACT(REPLACE(ALERT_RANGE, '&nbsp;', ' '), 'Min:\\s*([\\d.]+)', 1) AS DOUBLE) AS range_min,
  TRY_CAST(REGEXP_EXTRACT(REPLACE(ALERT_RANGE, '&nbsp;', ' '), 'Max:\\s*([\\d.]+)', 1) AS DOUBLE) AS range_max,
  TRY_CAST(SERVER_DT AS TIMESTAMP) AS server_timestamp,
  TRY_CAST(UID AS INT) AS uid,
  TRIM(OTHER_2) AS parameter_name
FROM workspace.scada_flow_2026.tb_report_alert

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.scada_cleaned.totalizer AS
SELECT
  UID AS uid, REPORT_DATE AS report_date, TRIM(IMEI_NUMBER) AS imei,
  TRY_CAST(START_DAY AS DOUBLE) AS start_day_reading,
  TRY_CAST(END_DAY AS DOUBLE) AS end_day_reading,
  TRY_CAST(END_DAY AS DOUBLE) - TRY_CAST(START_DAY AS DOUBLE) AS daily_consumption,
  CASE
    WHEN UPPER(TRIM(PNAME)) LIKE '%TOTAL%MLD%' THEN 'Total MLD'
    WHEN UPPER(TRIM(PNAME)) LIKE '%TOTALIZER%M3%' THEN 'Totalizer (m3)'
    WHEN UPPER(TRIM(PNAME)) LIKE '%TOTAL%M3%' THEN 'Total (m3)'
    WHEN UPPER(TRIM(PNAME)) LIKE '%TOTAL%KL%' THEN 'Total (KL)'
    ELSE TRIM(PNAME)
  END AS parameter_name,
  TRIM(OTHER_3) AS unit, SERVER_DT AS server_timestamp
FROM workspace.scada_flow_2026.tb_report_totalizer
WHERE TRY_CAST(END_DAY AS DOUBLE) IS NOT NULL

In [0]:
%sql
-- Top devices by reading count
SELECT
  d.device_name, d.location_name, d.site_code, d.status,
  COUNT(*) AS total_readings,
  MIN(s.data_timestamp) AS first_reading,
  MAX(s.data_timestamp) AS last_reading
FROM workspace.scada_cleaned.sensor_log s
JOIN workspace.scada_cleaned.device_list d ON s.imei = d.imei
GROUP BY 1,2,3,4
ORDER BY total_readings DESC
LIMIT 20

In [0]:
%sql
SELECT
  DATE_TRUNC('month', data_timestamp) AS month,
  COUNT(*) AS total_readings,
  COUNT(DISTINCT imei) AS active_devices
FROM workspace.scada_cleaned.sensor_log
GROUP BY 1 ORDER BY 1

In [0]:
%sql
SELECT
  d.device_name, d.location_name, a.alert_type,
  COUNT(*) AS alert_count
FROM workspace.scada_cleaned.alerts a
JOIN workspace.scada_cleaned.device_list d ON a.uid = d.uid
GROUP BY 1,2,3
ORDER BY alert_count DESC
LIMIT 20

In [0]:
%sql
WITH last_seen AS (
  SELECT imei, MAX(data_timestamp) AS last_reading
  FROM workspace.scada_cleaned.sensor_log GROUP BY imei
)
SELECT
  d.device_name, d.location_name, d.site_code,
  ls.last_reading,
  DATEDIFF(DAY, ls.last_reading, CURRENT_TIMESTAMP()) AS days_since_last
FROM workspace.scada_cleaned.device_list d
LEFT JOIN last_seen ls ON d.imei = ls.imei
WHERE d.status = 'Active'
  AND (ls.last_reading IS NULL OR DATEDIFF(DAY, ls.last_reading, CURRENT_TIMESTAMP()) > 7)
ORDER BY days_since_last DESC
LIMIT 30